In [3]:
import findspark
findspark.init()

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder. \
    appName("pyspark-1"). \
    getOrCreate()

### Read data

In [64]:
df = spark.read.csv("/dataset/nyc-jobs.csv", header=True)
df.printSchema()

root
 |-- Job ID: string (nullable = true)
 |-- Agency: string (nullable = true)
 |-- Posting Type: string (nullable = true)
 |-- # Of Positions: string (nullable = true)
 |-- Business Title: string (nullable = true)
 |-- Civil Service Title: string (nullable = true)
 |-- Title Code No: string (nullable = true)
 |-- Level: string (nullable = true)
 |-- Job Category: string (nullable = true)
 |-- Full-Time/Part-Time indicator: string (nullable = true)
 |-- Salary Range From: string (nullable = true)
 |-- Salary Range To: string (nullable = true)
 |-- Salary Frequency: string (nullable = true)
 |-- Work Location: string (nullable = true)
 |-- Division/Work Unit: string (nullable = true)
 |-- Job Description: string (nullable = true)
 |-- Minimum Qual Requirements: string (nullable = true)
 |-- Preferred Skills: string (nullable = true)
 |-- Additional Information: string (nullable = true)
 |-- To Apply: string (nullable = true)
 |-- Hours/Shift: string (nullable = true)
 |-- Work Locatio

### Sample function

In [48]:
def get_salary_frequency(df: DataFrame) -> list:
    row_list = df.select('Salary Frequency').distinct().collect()
    return [row['Salary Frequency'] for row in row_list]

### Example of test function

In [65]:
mock_data = [('A', 'Annual'), ('B', 'Daily')]
expected_result = ['Annual', 'Daily']

In [66]:
def test_get_salary_frequency(mock_data: list, 
                              expected_result: list,
                              schema: list = ['id', 'Salary Frequency']):  
    mock_df = spark.createDataFrame(data = mock_data, schema = schema)
    assert get_salary_frequency(mock_df) == expected_result

In [15]:
from pyspark.sql.functions import col
df = df.withColumn("Salary Range From",col("Salary Range From").cast("double")).withColumn("Salary Range To",col("Salary Range To").cast("double"))

In [16]:
df.show(5)


+------+--------------------+------------+--------------+--------------------+--------------------+-------------+-----+--------------------+-----------------------------+-----------------+---------------+----------------+--------------------+--------------------+--------------------+-------------------------+--------------------+----------------------+--------------------+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+--------------------+--------------------+
|Job ID|              Agency|Posting Type|# Of Positions|      Business Title| Civil Service Title|Title Code No|Level|        Job Category|Full-Time/Part-Time indicator|Salary Range From|Salary Range To|Salary Frequency|       Work Location|  Division/Work Unit|     Job Description|Minimum Qual Requirements|    Preferred Skills|Additional Information|            To Apply|         Hours/Shift|     Work Location 1| Recruitment Contact|Residency Require

In [17]:
#Top 10 job catogories
top_df = df.groupBy("Job Category").count().orderBy(col("count").desc()).show(10)

+--------------------+-----+
|        Job Category|count|
+--------------------+-----+
|Engineering, Arch...|  504|
|Technology, Data ...|  313|
|       Legal Affairs|  226|
|Public Safety, In...|  182|
|Building Operatio...|  181|
|Finance, Accounti...|  169|
|Administration & ...|  134|
|Constituent Servi...|  129|
|              Health|  125|
|Policy, Research ...|  124|
+--------------------+-----+
only showing top 10 rows



In [18]:
#Salary distribution per category

In [19]:
salary_df = df.groupBy("Job Category").agg({"Salary Range From":"avg","Salary Range To":"avg"}).show(5)

+--------------------+----------------------+--------------------+
|        Job Category|avg(Salary Range From)|avg(Salary Range To)|
+--------------------+----------------------+--------------------+
|Administration & ...|               90000.0|            100000.0|
|Health Policy, Re...|              113504.0|            143885.0|
|Administration & ...|               54100.0|             83981.0|
|Information Techn...|               68239.0|             85644.0|
|Finance, Accounti...|               55659.0|             70390.0|
+--------------------+----------------------+--------------------+
only showing top 5 rows



In [21]:
#Highest salary job per agency

In [23]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
wn = Window.partitionBy("Agency").orderBy(col("Salary Range To").desc())
highest_salary_df=df.withColumn("rank",row_number().over(wn)).filter(col("rank")==1).select("Agency","Business Title","Salary Range To").show()

+--------------------+--------------------+---------------+
|              Agency|      Business Title|Salary Range To|
+--------------------+--------------------+---------------+
|LANDMARKS PRESERV...|LANDMARKS PRESERV...|        64297.0|
|OFFICE OF COLLECT...|COLLEGE AIDE - CL...|          10.36|
|     FIRE DEPARTMENT|Senior Enterprise...|       144929.0|
|ADMIN FOR CHILDRE...|Director of Techn...|       156829.0|
|MANHATTAN COMMUNI...| Community Assistant|           19.0|
|      TAX COMMISSION|       CITY ASSESSOR|        90177.0|
|HRA/DEPT OF SOCIA...|EXECUTIVE DIRECTO...|       153017.0|
|TAXI & LIMOUSINE ...|Executive Directo...|       160000.0|
|EQUAL EMPLOY PRAC...|Director of Learn...|        72712.0|
|DEPARTMENT OF BUS...|EXECUTIVE DIRECTO...|       162014.0|
|DEPT OF DESIGN & ...|Associate Commiss...|       217244.0|
|TEACHERS RETIREME...|AGENCY ATTORNEY I...|        75760.0|
|DEPARTMENT OF COR...|Assistant Commiss...|       145000.0|
|FINANCIAL INFO SV...|Windows Technical.

In [24]:
# Avg salary per Agency(last 2 year)

In [31]:
from pyspark.sql.functions import current_date
avg_salary_df2= df.filter(datediff(current_date(),col("Posting Date")) <=730).groupBy("Agency").agg({"Salary Range From":"avg","Salary Range To":"avg"}).show()

+------+----------------------+--------------------+
|Agency|avg(Salary Range From)|avg(Salary Range To)|
+------+----------------------+--------------------+
+------+----------------------+--------------------+



In [33]:
df.select("Posting Date").show()

+--------------------+
|        Posting Date|
+--------------------+
|New York City res...|
|2012-01-26T00:00:...|
|                null|
|                null|
|2014-01-09T00:00:...|
|2014-01-09T00:00:...|
|Apply online with...|
|2013-12-20T00:00:...|
|                null|
|2014-06-26T00:00:...|
| ""2"" or ""3"" a...|
|New York City Res...|
|New York City Res...|
|2014-10-09T00:00:...|
|2014-10-09T00:00:...|
|2014-10-08T00:00:...|
|  mid-range computer|
| ""2"" or ""3"" a...|
|2014-11-18T00:00:...|
|           help desk|
+--------------------+
only showing top 20 rows



In [44]:
df_cleaned = df.withColumn("Year",year(to_date(col("Posting Date")))).filter(col("Year").isNotNull())

In [45]:
from pyspark.sql.functions import current_date,col,year
avg_salary_df2= df_cleaned.filter(col("Year")>=year(current_date())-2).groupBy("Agency").agg({"Salary Range From":"avg","Salary Range To":"avg"}).show()

+------+----------------------+--------------------+
|Agency|avg(Salary Range From)|avg(Salary Range To)|
+------+----------------------+--------------------+
+------+----------------------+--------------------+



In [ ]:
#Heighest paid skills

In [62]:
from pyspark.sql.functions import lower
skill_df=df.withColumn("Skills",lower(col("Preferred Skills"))).groupBy("Skills").agg({"Salary Range To": "avg"}).orderBy(col("avg(Salary Range To)").desc())

skill_df.show(10)

+--------------------+--------------------+
|              Skills|avg(Salary Range To)|
+--------------------+--------------------+
| and one year of ...|            225217.0|
|          commercial|  219745.66666666666|
|the deputy commis...|            218587.0|
|candidate must ha...|            217244.0|
| all candidates m...|            209585.0|
|â€¢	10+ years of ...|            208826.0|
|â€¢ integrity â€“...|            202744.0|
|extensive experie...|            198518.0|
|â€¢ expert knowle...|            194395.0|
| including making...|            194395.0|
+--------------------+--------------------+
only showing top 10 rows



In [67]:
df.select("Minimum Qual Requirements").show(5,truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Minimum Qual Requirements                                                                     

In [ ]:
#handling null values - data processing

In [69]:
df= df.fillna({"Salary Range From": 0,"Salary Range To":0})

In [ ]:
#Engineering Features -Average salary

In [70]:
df=df.withColumn("Avg Salary",(col("Salary Range From")+col("Salary Range To"))/2)

In [71]:
#Salary range


In [72]:
df=df.withColumn("Salary Range",col("Salary Range To")-col("Salary Range From"))

In [73]:
#job Age

In [74]:
df=df.withColumn("Job Age",datediff(current_date(),col("posting Date")))

In [75]:
#removing unnecessary columns

In [94]:
removing_df=df.drop("Job ID","Recruiment Contact","Additional Information").show()

+--------------------+------------+--------------+--------------------+--------------------+-------------+-----+--------------------+-----------------------------+-----------------+---------------+----------------+--------------------+--------------------+--------------------+-------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+--------------------+--------------------+----------+------------------+-------+
|              Agency|Posting Type|# Of Positions|      Business Title| Civil Service Title|Title Code No|Level|        Job Category|Full-Time/Part-Time indicator|Salary Range From|Salary Range To|Salary Frequency|       Work Location|  Division/Work Unit|     Job Description|Minimum Qual Requirements|    Preferred Skills|            To Apply|         Hours/Shift|     Work Location 1| Recruitment Contact|Residency Requirement|        Posting D

In [77]:
#writing the processed data

In [91]:
df.show()

+--------------------+------------+--------------+--------------------+--------------------+-------------+-----+--------------------+-----------------------------+-----------------+---------------+----------------+--------------------+--------------------+--------------------+-------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+--------------------+--------------------+----------+------------------+-------+
|              Agency|Posting Type|# Of Positions|      Business Title| Civil Service Title|Title Code No|Level|        Job Category|Full-Time/Part-Time indicator|Salary Range From|Salary Range To|Salary Frequency|       Work Location|  Division/Work Unit|     Job Description|Minimum Qual Requirements|    Preferred Skills|            To Apply|         Hours/Shift|     Work Location 1| Recruitment Contact|Residency Requirement|        Posting D

In [87]:
df.write.mode("overwrite").csv("/output/processed_jobs")